In [1]:
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
import os

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

from preprocessing.cleaner import CombustibilDataCleaner

In [3]:
combustibil = CombustibilDataCleaner()
combustibil.read_csv("merged", "combustibil_data.csv")
combustibil.reapply_types()

df = combustibil.get_data().sort_values("IDAlimentare").reset_index(drop=True)
print(f"{len(df)} transactions, {df['NumarInmatriculare'].nunique()} vehicles")
df.head()

2587 transactions, 55 vehicles


,IDAlimentare,Furnizor,Data,NumarInmatriculare,Produs,Cantitate,PretUnitar,Valoare,IDMasina
0,2,OMV,2025-04-01 08:21:00,CJ18TMI,OMV Diesel,73.08,6.3025,460.5882,1677
1,4,OMV,2025-04-08 08:49:00,CJ18TMI,OMV Diesel,70.51,6.1597,434.3193,1677
2,5,OMV,2025-04-11 10:29:00,CJ18TMI,OMV Diesel,74.92,6.1008,457.0756,1677
3,7,OMV,2025-04-15 15:10:00,CJ18TMI,OMV Diesel,74.84,6.0168,450.2941,1677
4,8,Petrom,2025-04-07 20:21:00,CJ59MTI,Motorina Extra,48.08,6.4537,310.2941,1516


In [4]:
df.describe()
# 1.26 39.02 46.25 64.125 252.16

,IDAlimentare,Data,Cantitate,PretUnitar,Valoare,IDMasina
count,2587.000000,2587,2587.000000,2587.000000,2587.000000,2587.000000
mean,3055.288365,2025-10-24 21:18:11.311944,51.239892,6.736440,345.423129,1343.506765
min,2.000000,2025-04-01 00:00:00,1.260000,5.748000,7.563000,26.000000
25%,1369.500000,2025-07-16 04:10:30,39.020000,6.281000,255.995000,1217.000000
50%,2816.000000,2025-10-23 10:08:00,46.250000,6.570000,309.752000,1314.000000
75%,4335.500000,2026-02-05 16:45:32.500000,64.125000,6.942000,429.387250,1516.000000
max,6640.000000,2026-05-15 21:21:00,252.160000,9.198400,1661.297500,1707.000000
std,1929.807614,NaN,20.596939,0.685212,142.472698,291.568776


In [5]:
## Many-valued context (ToscanaJ)

# Two exports:

# 1. **Transactions** — one object per refuel (`Id` = `IDAlimentare`), 2587 rows
# 2. **Vehicles** — one object per plate (`Id` = `NumarInmatriculare`), 55 rows

In [6]:
STANDARD_PRODUCTS = {
    "OMV Diesel", "Motorina standard", "Motorina Evo D", "EFIX MOTORINA 51",
}
PREMIUM_PRODUCTS = {
    "OMV MM Diesel", "OMV MM Diesel Artic", "Motorina Extra",
    "Motorina Evo D Plus", "EFIX S MOTORINA 55", "OMV MMotion95",
}
GASOLINE_PRODUCTS = {"Benzina Evo 95"}


def quartile_cutoffs(series):
    q25, q50, q75 = series.quantile([0.25, 0.50, 0.75])
    return series.min(), q25, q50, q75, series.max()


def bin_by_quartiles(value, vmin, q25, q50, q75, vmax):
    """Ordinal scale 1-4 for ToscanaJ: 1=min-Q25, 2=Q25-Q50, 3=Q50-Q75, 4=Q75-max."""
    if value <= q25:
        return 1
    if value <= q50:
        return 2
    if value <= q75:
        return 3
    return 4


PRICE_MIN, PRICE_Q25, PRICE_Q50, PRICE_Q75, PRICE_MAX = quartile_cutoffs(df["PretUnitar"])
COST_MIN, COST_Q25, COST_Q50, COST_Q75, COST_MAX = quartile_cutoffs(df["Valoare"])

# Transaction FillSize: domain cutoffs from EDA (price-sensitivity plot)
FILL_TOPUP_MAX = 25   # below main cluster
FILL_TYPICAL_MID = 50  # center of conglomerate
FILL_TYPICAL_MAX = 80  # upper bound before large/outlier fills

print("Transaction FillSize (L): 1 <=", FILL_TOPUP_MAX, "| 2 <=", FILL_TYPICAL_MID, "| 3 <=", FILL_TYPICAL_MAX, "| 4 >", FILL_TYPICAL_MAX)
print("UnitPrice scale (RON/L):  1 <=", round(PRICE_Q25, 2), "| 2 <=", round(PRICE_Q50, 2), "| 3 <=", round(PRICE_Q75, 2), "| 4 <=", round(PRICE_MAX, 2))
print("TotalCost scale (RON):    1 <=", round(COST_Q25, 2), "| 2 <=", round(COST_Q50, 2), "| 3 <=", round(COST_Q75, 2), "| 4 <=", round(COST_MAX, 2))


def product_tier(produs):
    if produs in PREMIUM_PRODUCTS:
        return "Premium"
    if produs in GASOLINE_PRODUCTS:
        return "Gasoline"
    return "Standard"


def fill_size_domain(liters):
    if liters <= FILL_TOPUP_MAX:
        return 1
    if liters <= FILL_TYPICAL_MID:
        return 2
    if liters <= FILL_TYPICAL_MAX:
        return 3
    return 4


fill_size_transaction = fill_size_domain


def supplier_loyalty_code(pct):
    if pct > 90:
        return 3  # Exclusive
    if pct >= 60:
        return 2  # Preferential
    return 1  # Opportunistic


def supplier_loyalty(pct):
    return {1: "Opportunistic", 2: "Preferential", 3: "Exclusive"}[supplier_loyalty_code(pct)]


SUPPLIER_CODE = {"MOL": 1, "OMV": 2, "Petrom": 3, "Rompetrol": 4}
PRODUCT_TIER_CODE = {"Standard": 1, "Premium": 2, "Gasoline": 3}
DAY_TYPE_CODE = {"Weekday": 1, "Weekend": 2}
TIME_SLOT_CODE = {"Morning": 1, "Afternoon": 2, "Evening": 3, "Unknown": 4}
QUARTER_CODE = {"2025Q2": 1, "2025Q3": 2, "2025Q4": 3, "2026Q1": 4, "2026Q2": 5}


def unit_price_band(price):
    return bin_by_quartiles(price, PRICE_MIN, PRICE_Q25, PRICE_Q50, PRICE_Q75, PRICE_MAX)


def total_cost_band(value):
    return bin_by_quartiles(value, COST_MIN, COST_Q25, COST_Q50, COST_Q75, COST_MAX)


def day_type(ts):
    return "Weekday" if ts.dayofweek < 5 else "Weekend"


def time_slot(ts):
    hour = ts.hour
    if hour == 0:
        return "Unknown"
    if 6 <= hour < 12:
        return "Morning"
    if 12 <= hour < 18:
        return "Afternoon"
    return "Evening"

Transaction FillSize (L): 1 <= 25 | 2 <= 50 | 3 <= 80 | 4 > 80
UnitPrice scale (RON/L):  1 <= 6.28 | 2 <= 6.57 | 3 <= 6.94 | 4 <= 9.2
TotalCost scale (RON):    1 <= 256.0 | 2 <= 309.75 | 3 <= 429.39 | 4 <= 1661.3


In [7]:
# vehicle cluster labels (same logic as clustering.ipynb)
PREMIUM_FOR_CLUSTER = PREMIUM_PRODUCTS
FEATURES = [
    "n_refills", "total_liters", "total_spend", "avg_fill",
    "n_suppliers", "loyalty_pct", "premium_share",
]


def top_supplier_share(group):
    return 100 * group["Furnizor"].value_counts().max() / len(group)


vehicle_profiles = (
    df.groupby("IDMasina")
    .apply(
        lambda g: pd.Series({
            "NumarInmatriculare": g["NumarInmatriculare"].iloc[0],
            "n_refills": len(g),
            "total_liters": g["Cantitate"].sum(),
            "total_spend": g["Valoare"].sum(),
            "avg_fill": g["Cantitate"].mean(),
            "n_suppliers": g["Furnizor"].nunique(),
            "loyalty_pct": top_supplier_share(g),
            "premium_share": g["Produs"].isin(PREMIUM_FOR_CLUSTER).mean() * 100,
        }),
        include_groups=False,
    )
    .reset_index()
)

X_scaled = StandardScaler().fit_transform(vehicle_profiles[FEATURES])
vehicle_profiles["cluster_kmeans"] = KMeans(
    n_clusters=3, random_state=42, n_init=10
).fit_predict(X_scaled)

vehicle_profiles["Cluster"] = vehicle_profiles["cluster_kmeans"] + 1
vehicle_profiles["SupplierLoyalty"] = vehicle_profiles["loyalty_pct"].apply(supplier_loyalty_code)

cluster_summary = (
    vehicle_profiles.groupby("Cluster")[FEATURES]
    .mean()
    .round(1)
)
print(cluster_summary)


cluster_sizes = vehicle_profiles["Cluster"].value_counts().sort_index()

print("Cluster sizes:", cluster_sizes.to_dict())

vehicle_lookup = vehicle_profiles.set_index("IDMasina")[
    ["NumarInmatriculare", "Cluster", "SupplierLoyalty"]
]
vehicle_lookup.head()

         n_refills  total_liters  total_spend  avg_fill  n_suppliers  \
Cluster                                                                
1             23.2        1068.4       7147.2      49.8          1.3   
2             62.8        3850.6      25948.5      65.8          2.8   
3             72.4        3106.8      21061.9      43.5          1.2   

         loyalty_pct  premium_share  
Cluster                              
1               92.7           24.3  
2               56.2           34.5  
3               96.8           52.1  
Cluster sizes: {1: 25, 2: 17, 3: 13}


c:\Users\Tudor\anaconda3\envs\ml-standard-env\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


,NumarInmatriculare,Cluster,SupplierLoyalty
IDMasina,,,
26,CJ58MTI,2,1
1129,CJ41MTI,1,3
1131,CJ52MTI,1,3
1168,CJ40MTI,1,2
1171,CJ22MTI,2,1


In [8]:
def label_cluster(row):
    if row["loyalty_pct"] >= 90 and row["premium_share"] >= 40:
        return "Premium loyal"
    if row["n_suppliers"] >= 2 and row["loyalty_pct"] < 80:
        return "Multi-brand active"
    return "Low-volume loyal"

cluster_summary["label"] = cluster_summary.apply(label_cluster, axis=1)
label_map = cluster_summary["label"].to_dict()

for cid, row in cluster_summary.iterrows():
    print(f"Cluster {cid} — {row['label']} (n={cluster_sizes[cid]}):")
    print(f"  refills={row.n_refills}, liters={row.total_liters}, loyalty={row.loyalty_pct}%, premium={row.premium_share}%")


Cluster 1 — Low-volume loyal (n=25):
  refills=23.2, liters=1068.4, loyalty=92.7%, premium=24.3%
Cluster 2 — Multi-brand active (n=17):
  refills=62.8, liters=3850.6, loyalty=56.2%, premium=34.5%
Cluster 3 — Premium loyal (n=13):
  refills=72.4, liters=3106.8, loyalty=96.8%, premium=52.1%


In [9]:
transaction_context = pd.DataFrame({
    "Id": df["IDAlimentare"],
    "Supplier": df["Furnizor"].map(SUPPLIER_CODE),
    "ProductTier": df["Produs"].apply(product_tier).map(PRODUCT_TIER_CODE),
    "FillSize": df["Cantitate"].apply(fill_size_transaction),
    "UnitPrice": df["PretUnitar"].apply(unit_price_band),
    "TotalCost": df["Valoare"].apply(total_cost_band),
    "DayType": df["Data"].apply(day_type).map(DAY_TYPE_CODE),
    "TimeSlot": df["Data"].apply(time_slot).map(TIME_SLOT_CODE),
    "Quarter": df["Data"].dt.to_period("Q").astype(str).map(QUARTER_CODE),
}).astype({
    "Id": int,
    "Supplier": int,
    "ProductTier": int,
    "FillSize": int,
    "UnitPrice": int,
    "TotalCost": int,
    "DayType": int,
    "TimeSlot": int,
    "Quarter": int,
})

transaction_context.head(10)

,Id,Supplier,ProductTier,FillSize,UnitPrice,TotalCost,DayType,TimeSlot,Quarter
0,2,2,1,3,2,4,1,1,1
1,4,2,1,3,1,4,1,1,1
2,5,2,1,3,1,4,1,1,1
3,7,2,1,3,1,4,1,2,1
4,8,3,2,2,2,3,1,3,1
5,9,3,2,2,2,2,1,3,1
6,15,3,1,3,1,4,2,2,1
7,16,2,2,3,3,4,1,3,1
8,24,2,1,2,2,2,1,2,1
9,25,3,1,2,1,2,1,1,1


In [ ]:
def dominant_value(series):
    return series.value_counts().index[0]


vehicle_attrs = (
    df.groupby("IDMasina")
    .apply(
        lambda g: pd.Series({
            "Supplier": dominant_value(g["Furnizor"]),
            "ProductTier": dominant_value(g["Produs"].apply(product_tier)),
            "mean_fill": g["Cantitate"].mean(),
            "mean_price": g["PretUnitar"].mean(),
            "total_spend": g["Valoare"].sum(),
            "DayType": dominant_value(g["Data"].apply(day_type)),
            "TimeSlot": dominant_value(g["Data"].apply(time_slot)),
            "Quarter": dominant_value(g["Data"].dt.to_period("Q").astype(str)),
        }),
        include_groups=False,
    )
    .reset_index()
    .merge(vehicle_lookup.reset_index(), on="IDMasina")
)

VP_MIN, VP_Q25, VP_Q50, VP_Q75, VP_MAX = quartile_cutoffs(vehicle_attrs["mean_price"])
VS_MIN, VS_Q25, VS_Q50, VS_Q75, VS_MAX = quartile_cutoffs(vehicle_attrs["total_spend"])

print("Vehicle FillSize (L): 1 <=", FILL_TOPUP_MAX, "| 2 <=", FILL_TYPICAL_MID, "| 3 <=", FILL_TYPICAL_MAX, "| 4 >", FILL_TYPICAL_MAX)
print("Vehicle UnitPrice (RON/L): 1 <=", round(VP_Q25, 2), "| 2 <=", round(VP_Q50, 2), "| 3 <=", round(VP_Q75, 2), "| 4 <=", round(VP_MAX, 2))
print("Vehicle TotalCost (RON):   1 <=", round(VS_Q25, 2), "| 2 <=", round(VS_Q50, 2), "| 3 <=", round(VS_Q75, 2), "| 4 <=", round(VS_MAX, 2))

vehicle_context = pd.DataFrame({
    "Id": vehicle_attrs["IDMasina"],
    "Supplier": vehicle_attrs["Supplier"].map(SUPPLIER_CODE),
    "ProductTier": vehicle_attrs["ProductTier"].map(PRODUCT_TIER_CODE),
    "FillSize": vehicle_attrs["mean_fill"].apply(fill_size_domain),
    "UnitPrice": vehicle_attrs["mean_price"].apply(
        lambda x: bin_by_quartiles(x, VP_MIN, VP_Q25, VP_Q50, VP_Q75, VP_MAX)
    ),
    "TotalCost": vehicle_attrs["total_spend"].apply(
        lambda x: bin_by_quartiles(x, VS_MIN, VS_Q25, VS_Q50, VS_Q75, VS_MAX)
    ),
    "DayType": vehicle_attrs["DayType"].map(DAY_TYPE_CODE),
    "TimeSlot": vehicle_attrs["TimeSlot"].map(TIME_SLOT_CODE),
    "Quarter": vehicle_attrs["Quarter"].map(QUARTER_CODE),
    "Cluster": vehicle_attrs["Cluster"],
    "SupplierLoyalty": vehicle_attrs["SupplierLoyalty"],
}).astype({
    "Id": int,
    "Supplier": int,
    "ProductTier": int,
    "FillSize": int,
    "UnitPrice": int,
    "TotalCost": int,
    "DayType": int,
    "TimeSlot": int,
    "Quarter": int,
    "Cluster": int,
    "SupplierLoyalty": int,
})

vehicle_context.head(10)

Vehicle FillSize (L): 1 <= 25 | 2 <= 50 | 3 <= 80 | 4 > 80
Vehicle UnitPrice (RON/L): 1 <= 6.58 | 2 <= 6.71 | 3 <= 6.9 | 4 <= 8.56
Vehicle TotalCost (RON):   1 <= 8293.76 | 2 <= 15237.72 | 3 <= 23225.62 | 4 <= 39728.64
<class 'pandas.DataFrame'>
RangeIndex: 55 entries, 0 to 54
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Id               55 non-null     int64
 1   Supplier         55 non-null     int64
 2   ProductTier      55 non-null     int64
 3   FillSize         55 non-null     int64
 4   UnitPrice        55 non-null     int64
 5   TotalCost        55 non-null     int64
 6   DayType          55 non-null     int64
 7   TimeSlot         55 non-null     int64
 8   Quarter          55 non-null     int64
 9   Cluster          55 non-null     int64
 10  SupplierLoyalty  55 non-null     int64
dtypes: int64(11)
memory usage: 4.9 KB


In [11]:
for name, ctx in [("Transactions", transaction_context), ("Vehicles", vehicle_context)]:
    print(f"=== {name} ===")
    print(f"Objects: {len(ctx)}")
    print(f"Attributes: {len(ctx.columns) - 1}\n")
    for col in ctx.columns[1:]:
        print(f"{col}: {ctx[col].nunique()} values")
    print()

=== Transactions ===
Objects: 2587
Attributes: 8

Supplier: 4 values
ProductTier: 3 values
FillSize: 4 values
UnitPrice: 4 values
TotalCost: 4 values
DayType: 2 values
TimeSlot: 4 values
Quarter: 5 values

=== Vehicles ===
Objects: 55
Attributes: 10

Supplier: 4 values
ProductTier: 2 values
FillSize: 3 values
UnitPrice: 4 values
TotalCost: 4 values
DayType: 1 values
TimeSlot: 3 values
Quarter: 5 values
Cluster: 3 values
SupplierLoyalty: 3 values



In [ ]:
output_dir = os.path.join(project_root, "dataset", "toscanaj")
os.makedirs(output_dir, exist_ok=True)

transaction_path = os.path.join(output_dir, "combustibil_context.csv")
vehicle_path = os.path.join(output_dir, "combustibil_vehicle_context.csv")

transaction_context.to_csv(transaction_path, index=False)
vehicle_context.to_csv(vehicle_path, index=False)

print(f"Saved {len(transaction_context)} rows to {transaction_path}")
print(f"Saved {len(vehicle_context)} rows to {vehicle_path}")

In [ ]:
transaction_context.drop(columns=["Id"], inplace=True)
transaction_path_conexp = os.path.join(output_dir, "combustibil_context_conexp.csv")
transaction_context.sample(25).to_csv(transaction_path_conexp, index=False)

### Scale reference — `combustibil_vehicle_context.csv`

All attributes are **integers** for ToscanaJ.

| Attribute | Type | Values | Meaning |
|-----------|------|--------|---------|
| **Id** | object key | IDMasina | Vehicle identifier |
| **Supplier** | nominal | 1=MOL, 2=OMV, 3=Petrom, 4=Rompetrol | Dominant supplier |
| **ProductTier** | nominal | 1=Standard, 2=Premium, 3=Gasoline | Dominant fuel grade |
| **FillSize** | ordinal | 1, 2, 3, 4 | Mean fill (L): 1<=25, 2<=50, 3<=80, 4>80 |
| **UnitPrice** | ordinal | 1, 2, 3, 4 | Mean RON/L quartiles (55 vehicles) |
| **TotalCost** | ordinal | 1, 2, 3, 4 | Total spend quartiles (55 vehicles) |
| **DayType** | nominal | 1=Weekday, 2=Weekend | Dominant refuel day |
| **TimeSlot** | nominal | 1=Morning, 2=Afternoon, 3=Evening, 4=Unknown | Dominant refuel hour |
| **Quarter** | nominal | 1=2025Q2 ... 5=2026Q2 | Dominant activity quarter |
| **Cluster** | nominal | 1, 2, 3 | K-Means vehicle profile |
| **SupplierLoyalty** | ordinal | 1=Opportunistic, 2=Preferential, 3=Exclusive | Top-supplier share |

See next cell for the **transaction** scale legend (`combustibil_context.csv`).

### Scale reference — `combustibil_context.csv` (transactions)

All attributes are **integers**. Ordinal cutoffs are printed in cell 5.

| Attribute | Type | Code | Meaning |
|-----------|------|------|---------|
| **Id** | object key | IDAlimentare | Refuel transaction id |
| **Supplier** | nominal | 1=MOL, 2=OMV, 3=Petrom, 4=Rompetrol | Supplier on this refuel |
| **ProductTier** | nominal | 1=Standard, 2=Premium, 3=Gasoline | Fuel grade |
| **FillSize** | ordinal | 1-4 | 1: <= 25 L, 2: 25-50L, 3: 50-80L, 4: >80L  |
| **UnitPrice** | ordinal | 1-4 | PretUnitar quartiles (RON/L) |
| **TotalCost** | ordinal | 1-4 | Valoare quartiles (RON) |
| **DayType** | nominal | 1=Weekday, 2=Weekend | Day of refuel |
| **TimeSlot** | nominal | 1=Morning, 2=Afternoon, 3=Evening, 4=Unknown | Hour of refuel |
| **Quarter** | nominal | 1=2025Q2 ... 5=2026Q2 | Calendar quarter |